*** TO DO FOR CATAN: ***
RAINBOW: 
    1. vs Random
    2. vs Weighted Random
    3. vs MTCS
    4. vs Victory Point
    5. vs AlphaBeta
Masked PPO the same 
NFSP 
MuZero

MUZERO

In [ ]:
import copy
import sys


sys.path.append("../../../")

import torch

from agents.catan_player_wrapper import CatanPlayerWrapper

from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from modules.world_models.muzero_world_model import MuzeroWorldModel

from catanatron import Game, RandomPlayer, Color
from catanatron.players.mcts import MCTSPlayer
from catanatron.players.minimax import AlphaBetaPlayer
from catanatron.players.playouts import GreedyPlayoutsPlayer
from catanatron.players.search import VictoryPointPlayer
from catanatron.players.weighted_random import WeightedRandomPlayer
from catanatron.players.value import ValueFunctionPlayer
from utils.wrappers import record_video_wrapper
from stats.stats import StatTracker

from configs.games.catan import CatanConfig
from custom_gym_envs.envs.catan import (
    env as catan_env,
    CatanAECEnv,
)

from torch.optim import Adam, SGD

env = CatanConfig().make_env()
env = record_video_wrapper(copy.deepcopy(env), "./videos", 1)
params = {
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
    },
    # 1. Architecture (Robust Image processing via ResNet)
    "representation_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "dynamics_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "prediction_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    # 2. Heads (Distributional for stability)
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "categorical"},
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Architecture
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
        "residual_layers": [(24, 3, 1)],
    },
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # 3. Optimization
    # "learning_rate": 0.001,
    "optimizer": torch.optim.Adam,
    "weight_decay": 0.0001,
    "lr_ratio": float("inf"),  # Scale representation gradients
    "minibatch_size": 8,  # Scaled for stability
    "training_steps": 100000,
    # 4. Search / MCTS
    "num_simulations": 48,  # Increased for better planning
    "root_dirichlet_alpha": 0.3,
    "discount_factor": 0.999,  # Handle long sequences in Catan
    # 5. EfficientZero Features (Standard for robust MuZero)
    "reanalyze_ratio": 0.0,  # Maximize data efficiency
    "consistency_loss_factor": 0.0,
    # 6. Replay Buffer
    "replay_buffer_size": 10000,
    "min_replay_buffer_size": 32,
    "n_step": 250,
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "linear",
        "initial": 0.0,
        "final": 0.0,
        "decay_steps": 100000,
    },
    # 7. Misc
    "value_prefix": False,
    "record_video": True,
    "record_video_interval": 100,
    "num_workers": 4,
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "search_batch_size": 0,
    "use_virtual_mean": True,
    "search_backend": "aos",
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [20],  # after 5 moves in episode
        "values": [1.0, 0.0],  # start at 1.0, switch to 0.0 after step 5
    },
    "transfer_interval": 100,
    "gumbel": True,
}
game_config = CatanConfig()
config = MuZeroConfig(config_dict=params, game_config=game_config)


trainer = MuZeroTrainer(
    config=config,
    env=env,
    device=torch.device("cpu"),
    name="catan_bandit",
    stats=StatTracker("catan_bandit"),
    test_agents=[
        CatanPlayerWrapper(RandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(WeightedRandomPlayer, Color.BLUE),
        # CatanPlayerWrapper(AlphaBetaPlayer, Color.BLUE),
    ],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 500
trainer.test_trials = 50

trainer.train()

In [ ]:
import copy
import sys
import time
import numpy as np
import torch
from torch.optim import Adam

# Adjust paths as necessary
sys.path.append("../../../")

from catanatron.players.playouts import GreedyPlayoutsPlayer
from catanatron.players.weighted_random import WeightedRandomPlayer
from catanatron.players.minimax import AlphaBetaPlayer

from agents.catan_player_wrapper import CatanPlayerWrapper
from agents.learners.imitation_learner import ImitationLearner
from modules.world_models.muzero_world_model import MuzeroWorldModel
from configs.agents.muzero import MuZeroConfig
from configs.games.catan import CatanConfig
from custom_gym_envs.envs.catan import CatanAECEnv
from stats.stats import StatTracker

# ==============================================================================
# 1. Configuration & Architecture definition
# ==============================================================================
params = {
    # Architecture (Robust Image processing via ResNet)
    "representation_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "dynamics_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    "prediction_backbone": {
        "type": "resnet",
        "filters": [32] * 2,
        "kernel_sizes": [3] * 2,
        "strides": [1] * 2,
        "norm_type": "batch",
    },
    # Heads
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "categorical"},
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        }
    },
    # Optimization & SL Parameters required by ImitationLearner
    "learning_rate": 0.001,
    "adam_epsilon": 1e-8,
    "optimizer": Adam,
    "weight_decay": 0.0001,
    "minibatch_size": 128,  # Increased for faster SL updates
    "replay_buffer_size": 50000,  # Large buffer for offline data
    "min_replay_buffer_size": 256,
}

game_config = CatanConfig()
config = MuZeroConfig(config_dict=params, game_config=game_config)

# ==============================================================================
# 2. Environment, Expert, and Model Setup
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

env = game_config.make_env()

# Assuming AEC Env, get dimensions from the first agent
first_agent = env.possible_agents[0]
obs_dim = env.observation_space(first_agent).shape
num_actions = env.action_space(first_agent).n
obs_dtype = torch.float32

# Instantiate your full MuZero architecture to test its capacity
agent_network = MuzeroWorldModel(
    config=config, observation_dimensions=obs_dim, num_actions=num_actions
).to(device)

# Instantiate the Imitation Learner to handle buffer and CrossEntropyLoss
learner = ImitationLearner(
    config=config,
    agent_network=agent_network,
    device=device,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=obs_dtype,
)

# We use a wrapper around an expert. (Replace GreedyPlayoutsPlayer with what works best)
expert_agent = CatanPlayerWrapper(GreedyPlayoutsPlayer, None)

# ==============================================================================
# 3. Training Loop (Collect & Train)
# ==============================================================================
num_epochs = 50
episodes_per_epoch = 5
training_steps_per_epoch = 100

stats = StatTracker("catan_imitation")
stats._init_key("loss")
stats._init_key("accuracy")

print("Starting Imitation Learning test for MuZero Architecture...")

for epoch in range(num_epochs):
    # --- A. Data Collection ---
    print(f"\nEpoch {epoch+1}/{num_epochs} - Collecting data...")
    collected_transitions = 0

    for _ in range(episodes_per_epoch):
        env.reset()
        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()

            if termination or truncation:
                env.step(None)
                continue

            # 1. Get Expert Action
            # Some wrappers might need specific arguments, adjust if your wrapper differs
            expert_action = expert_agent.predict(obs, info, env)
            if isinstance(expert_action, tuple):
                expert_action = expert_action[
                    0
                ]  # Handle if wrapper returns (action, info)

            # 2. Create One-Hot Target
            target_policy = torch.zeros(num_actions)
            target_policy[expert_action] = 1.0

            # 3. Store in Learner
            learner.store(
                observation=obs,
                legal_moves=info.get("valid_actions", info.get("legal_moves", [])),
                target_policy=target_policy,
            )

            # 4. Step environment
            env.step(expert_action)
            collected_transitions += 1

    print(
        f"Collected {collected_transitions} transitions. Buffer size: {learner.replay_buffer.size()}"
    )

    # --- B. Training ---
    if learner.replay_buffer.size() > config.min_replay_buffer_size:
        print(f"Training for {training_steps_per_epoch} steps...")

        epoch_losses = []
        epoch_accuracies = []

        for _ in range(training_steps_per_epoch):
            # 1. Standard Learner step (computes Cross Entropy against expert target)
            step_results = learner.step(stats=stats)
            if step_results and "imitation_loss" in step_results:
                epoch_losses.append(step_results["imitation_loss"])

            # 2. Compute explicit Accuracy to see if it's learning
            batch = learner.replay_buffer.sample(config.minibatch_size)
            obs_batch = batch["observations"].to(device)
            targets = batch["target_policies"].to(device)

            with torch.no_grad():
                inf_out = agent_network.obs_inference(obs_batch)
                predictions = inf_out.policy.probs  # Output probabilities
                predicted_actions = predictions.argmax(dim=-1)
                target_actions = targets.argmax(dim=-1)
                acc = (predicted_actions == target_actions).float().mean().item()
                epoch_accuracies.append(acc)

        avg_loss = np.mean(epoch_losses) if epoch_losses else float("inf")
        avg_acc = np.mean(epoch_accuracies) if epoch_accuracies else 0.0

        print(f"--> Loss: {avg_loss:.4f} | Accuracy: {avg_acc*100:.2f}%")

print(
    "\nFinished Testing. If Accuracy goes up steadily, your MuZero observation processing and policy head are expressive enough!"
)